# Delta Prompting Attack v5.1 - Contrastive Why-Interrogation

Transfer attack using a local whitebox proxy model to attack closed models via OpenRouter.

**Key Innovation (v5.1)**: Contrastive steering using "why refuse?" (TARGET) vs "why comply?" (Venice):
```
steering_direction = comply_acts - refuse_acts
```

**Requirements:**
- OpenRouter API key (for target model)
- Venice API key (for compliant suffix + "why comply" responses)
- Local GPU with ~12-16GB VRAM (for Llama 3.1 8B 4-bit)

In [1]:
# ============================================================
# ENVIRONMENT SETUP
# ============================================================

import os
import sys
from dotenv import load_dotenv

load_dotenv()

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/JB_mech_outputs/delta_prompting'
    
    if not os.path.exists('JB_mech'):
        !git clone https://github.com/ChuloIva/JB_mech.git
    %cd JB_mech
    !pip install -q transformers peft bitsandbytes accelerate openai datasets scikit-learn baukit
    
    PROJECT_ROOT = os.getcwd()
else:
    PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
    if os.path.basename(os.getcwd()) == 'notebooks':
        PROJECT_ROOT = os.path.dirname(os.getcwd())
    else:
        PROJECT_ROOT = os.getcwd()
    OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs/delta_prompting')

# Add paths
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'third_party/activation_oracles'))

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")

Project root: /Users/ivanculo/Desktop/Projects/JB_mech
Output dir: /Users/ivanculo/Desktop/Projects/JB_mech/outputs/delta_prompting


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

from jb_mech.delta_attack import DeltaAttackConfig

# Create configuration - modify values as needed
config = DeltaAttackConfig(
    # Target model (closed, via OpenRouter)
    target_model="openai/gpt-4o-mini",
    
    # Proxy model (local whitebox)
    # proxy_model_name="unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    # proxy_oracle_lora="adamkarvonen/checkpoints_latentqa_cls_past_lens_Llama-3_1-8B-Instruct",

    proxy_model_name = "Qwen/Qwen3-4B",
    proxy_oracle_lora = "adamkarvonen/checkpoints_latentqa_cls_past_lens_Qwen3-4B",
    
    # Attack settings
    max_iterations=10,
    why_iterations=4,
    
    # Contrastive WHY (v5.1 key feature)
    use_contrastive_why=True,
    contrastive_why_model="venice-uncensored",
    
    # Steering
    use_steering=True,
    steering_alpha=2.0,  # Positive for contrastive (comply - refuse)
    steering_method="pca",
    steering_layer_start=15,
    steering_layer_end=None,  # None = steer to end
    
    # GLP manifold (optional)
    use_glp_manifold=False,
    
    # Compliant suffix
    dynamic_compliant_suffix=True,
    dynamic_suffix_model="venice-uncensored",
    
    # Evaluator
    use_llm_evaluator=True,
    evaluator_model="google/gemma-3-27b-it:free",
    rewriter_model="arcee-ai/trinity-large-preview:free",
)

# Number of test prompts to run
N_TEST_PROMPTS = 3

# Print configuration
config.print_config()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4029486733.py, line 16)

In [ ]:
# ============================================================
# LOAD PROXY MODEL
# ============================================================

os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from huggingface_hub import login
from jb_mech.wrappers import ActivationOracleWrapper

# HuggingFace login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face")

torch.set_grad_enabled(False)

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\nLoading proxy model: {config.proxy_model_name}")

ao = ActivationOracleWrapper.from_pretrained(
    config.proxy_model_name,
    oracle_lora_path=config.proxy_oracle_lora,
    layer_percent=config.layer_percent,
)

print(f"\nProxy model loaded!")
print(f"  Capture layer: {ao.capture_layer} / {ao.num_layers}")

In [ ]:
# ============================================================
# RUN TRANSFER ATTACKS
# ============================================================

import json
import random
from datasets import load_dataset
from jb_mech.delta_attack import transfer_attack_loop, run_attack_batch

# Load test prompts from JailbreakBench
print("Loading test prompts from JailbreakBench...")
jbb = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")
all_harmful = list(jbb["harmful"])

# Sample test prompts
sampled_items = random.sample(all_harmful, min(N_TEST_PROMPTS, len(all_harmful)))
test_goals = [item["Goal"] for item in sampled_items]

print(f"\n{'='*60}")
print(f"ATTACK CONFIGURATION")
print(f"{'='*60}")
config.print_config()
print(f"{'='*60}")

print(f"\nTest goals ({len(test_goals)}):")
for goal in test_goals:
    print(f"  - {goal[:60]}...")

print("\n" + "="*60)
print("RUNNING ATTACKS")
print("="*60)

# Run attacks
results = run_attack_batch(
    goals=test_goals,
    ao=ao,
    config=config,
    output_dir=OUTPUT_DIR,
    checkpoint_every=2,
)

# Save final results
final_path = os.path.join(OUTPUT_DIR, "results_final.json")
with open(final_path, "w") as f:
    json.dump([r.__dict__ for r in results], f, indent=2, default=str)

# Summary
successes = sum(1 for r in results if r.success)
print(f"\n{'='*60}")
print(f"ATTACK COMPLETE")
print(f"{'='*60}")
print(f"Success rate: {successes}/{len(results)} ({100*successes/len(results):.0f}%)")
print(f"Results: {final_path}")
print("="*60)

In [ ]:
# ============================================================
# CLEANUP
# ============================================================

ao.cleanup()
print("Proxy model cleaned up.")
print(f"Results saved to: {OUTPUT_DIR}")